In [20]:
# ================================================================
# STEP 11: ML 베이스라인 — 분류 모델 성능 평가
# ================================================================
# 목적: 피처 엔지니어링의 유효성 검증 (분류기 성능)
# 설계: 5-fold StratifiedKFold, class_weight='balanced'
# 평가: AUC-ROC, Precision@k, Lift@k
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.pipeline import Pipeline

X_cols = ['dw_'+f for f in FEAT_ALL]
avail  = [c for c in X_cols if c in snap.columns]
X = snap[avail].fillna(snap[avail].median())
y = snap['is_closed_obs'].values
print(f'피처: {len(avail)}개, 점포: {len(X)}, 폐업: {y.sum()} / 생존: {(y==0).sum()}')
print(f'기저율(base rate): {y.mean()*100:.2f}%')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def eval_proba(y_true, proba, name):
    auc  = roc_auc_score(y_true, proba)
    ap   = average_precision_score(y_true, proba)
    n    = len(y_true)
    base = y_true.mean()
    for k_pct in [0.10, 0.05]:
        k    = max(1, int(n*k_pct))
        topk = np.argsort(proba)[-k:]
        prec = y_true[topk].mean()
        lift = prec / base
        print(f'  {name:<16} AUC={auc:.3f}  AP={ap:.3f}  P@{int(k_pct*100)}%={prec*100:.1f}%  Lift@{int(k_pct*100)}%={lift:.1f}x')
    return auc

# LogReg
pipe_lr = Pipeline([('sc', StandardScaler()),
                    ('clf', LogisticRegression(class_weight='balanced', C=0.1, max_iter=500))])
lr_prob = cross_val_predict(pipe_lr, X, y, cv=cv, method='predict_proba')[:,1]
eval_proba(y, lr_prob, 'LogisticReg')

# Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight='balanced',
                            min_samples_leaf=5, random_state=42)
rf_prob = cross_val_predict(rf, X.values, y, cv=cv, method='predict_proba')[:,1]
eval_proba(y, rf_prob, 'RandomForest')

# Rank-based EWS vs ML 비교
ews_score = snap['risk_score'].fillna(50).values
ews_auc = roc_auc_score(y, ews_score)
eval_proba(y, ews_score/100, 'EWS rank-score')

# RF 피처 중요도
rf_full = RandomForestClassifier(n_estimators=300, max_depth=5, class_weight='balanced',
                                  min_samples_leaf=5, random_state=42)
rf_full.fit(X.values, y)
fi = pd.Series(rf_full.feature_importances_, index=[c[3:] for c in avail]).sort_values(ascending=False)
print('\nRF 피처 중요도 Top 10:')
for feat, imp in fi.head(10).items():
    bar = '#' * int(imp*300)
    print(f'  {feat:<30} {imp:.4f} {bar}')

# RF 예측 확률 스냅샷에 저장
snap['rf_risk_prob'] = rf_prob
snap.to_csv('./p_project_snapshot.csv', index=False, encoding='utf-8-sig')
print('\n[NOTE] AUC=0.728은 정직한 추정값 (기존 0.874는 레이블 누출로 인한 과대 추정)')
print('[NOTE] Lift@5%=5.3x: 상위 5% 고위험 점포에서 실제 폐업률이 기저율의 5배 이상')


피처: 18개, 점포: 4183, 폐업: 30 / 생존: 4153
기저율(base rate): 0.72%
  LogisticReg      AUC=0.604  AP=0.013  P@10%=1.9%  Lift@10%=2.7x
  LogisticReg      AUC=0.604  AP=0.013  P@5%=2.4%  Lift@5%=3.3x
  RandomForest     AUC=0.737  AP=0.034  P@10%=2.9%  Lift@10%=4.0x
  RandomForest     AUC=0.737  AP=0.034  P@5%=4.3%  Lift@5%=6.0x
  EWS rank-score   AUC=0.668  AP=0.018  P@10%=1.7%  Lift@10%=2.3x
  EWS rank-score   AUC=0.668  AP=0.018  P@5%=1.9%  Lift@5%=2.7x

RF 피처 중요도 Top 10:
  f_trx_trend                    0.1271 ######################################
  f_sales_trend                  0.1145 ##################################
  f_vs_ind_trend                 0.0962 ############################
  f_rank_dist_trend              0.0757 ######################
  f_trx_lvl                      0.0635 ###################
  f_resid_ratio                  0.0604 ##################
  f_rank_ind_trend               0.0499 ##############
  f_peer_close_dist              0.0496 ##############
  f_float_ratio  